## Regression Analysis for Predicting Soil Thickness
Test run - in development

## Load libraries

In [ ]:
# load relevant libraries
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import math
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

## Prep/load training data

In [ ]:
# Load the training data CSV file compiled in `1-data_access.ipynb`
input_file = '../data/2-uw_layer1_trainingdata.csv'
training_data = pd.read_csv(input_file)

# Create a GeoDataFrame from the DataFrame
training_data = gpd.GeoDataFrame(training_data, geometry=gpd.points_from_xy(training_data.LONGITUDE, training_data.LATITUDE))
    
# Set the CRS to WGS84 (latitude and longitude)
training_data.set_crs(epsg=4326, inplace=True)

training_data

## Training data, preprocessing, and splits

In [ ]:
# Pull training data from dataframe
X = training_data[['MAPPED_UNIT','SLOPE','ROUGHNESS']]
y_1 = training_data['LAYER_THICKNESS_FT']

# Preprocessing to convert MAPPED_UNIT to a usable, non-string format
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(sparse_output=False), ['MAPPED_UNIT']),  # use OneHotEncoder for categorical features
        ('num', StandardScaler(), ['SLOPE', 'ROUGHNESS'])  # Standardize numerical features
    ])

# Prepare the preprocessed data to be incorporated into the data split
X_processed = preprocessor.fit_transform(X)

# Handling target for LAYER_THICKNESS_FT
scaler = StandardScaler()
y_1_scaled = scaler.fit_transform(y_1.values.reshape(-1, 1))

# Split the data into training and test sets
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X_processed, y_1_scaled, test_size=0.2, random_state=42)

# We need to scale the x variable too, otherwise the data training is very bad
#X_train_scaled = scaler.fit_transform(X_train_1)

print(X_train_1) ## may be an issue here because this gives a sparse row sparse matrix, but we're gonna keep going


## Model training

In [ ]:
# Define the kernel
kernel = C(1.0, (1e-2, 1e2)) * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 1e2))

# # Initialize GaussianProcessRegressor
gaussian_process = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, alpha=1e-6)

# Fit the model (scikit-learn handles optimization internally)
gaussian_process.fit(X_train_1, y_train_1)


In [ ]:
# Make model predictions
gaussian_process.predict(X_test_1, return_std=True)

# Unscale the data for visualizations
pred_unscaled = scaler.inverse_transform(y_1_scaled)

plt.hist(pred_unscaled)
